# Agentic Data Agent — Mount

Mounts the client databases (`client/db/db-1 … db-16`) and loads their BIRD-style
question–SQL pairs for text-to-SQL agent development, per `client/doc/README.md`.

- **Mount** = load each database's documentation and `queries.json` (no live DB needed).
- **BIRD pairs** = `(question, sql, description, evidence, expected_output)` per query.
- **Live execution** (optional) = PostgreSQL containers on ports 5436–5451
  (`./scripts/setup_docker.sh -a` from `client/`, or `docker/docker-compose.hardened.yml`
  from the repo root). Schema-only mode is the sanctioned quick test — queries may return 0 rows.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "scripts" / "agentic_mount.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

from agentic_mount import load_client_db, get_bird_pairs, get_pg_port

CLIENT_DB = ROOT / "client" / "db"
print(f"repo root: {ROOT}")
print(f"client db dir: {CLIENT_DB} (exists={CLIENT_DB.exists()})")

repo root: /home/user/db
client db dir: /home/user/db/client/db (exists=True)


## 1. Mount all databases

In [2]:
mounted = {n: load_client_db(CLIENT_DB, n) for n in range(1, 17)}

print(f"{'db':<7}{'queries':>8}{'docs':>7}{'pg port':>9}")
total = 0
for n, m in mounted.items():
    q = len(m["queries"]); total += q
    print(f"{m['db']:<7}{q:>8}{'yes' if m.get('docs') else 'no':>7}{get_pg_port(n):>9}")
print(f"{'total':<7}{total:>8}")

db      queries   docs  pg port
db-1         30    yes     5436
db-2         30    yes     5437
db-3         30    yes     5438
db-4         30    yes     5439
db-5         30    yes     5440
db-6         30    yes     5441
db-7         30    yes     5442
db-8         30    yes     5443
db-9         30    yes     5444
db-10        30    yes     5445
db-11        30    yes     5446
db-12        30    yes     5447
db-13        30    yes     5448
db-14        30    yes     5449
db-15        30    yes     5450
db-16        30    yes     5451
total       480


## 2. BIRD-style pairs (example: db-9)

In [3]:
pairs = get_bird_pairs(mounted[9], 9)
print(f"db-9 pairs: {len(pairs)}")
p = pairs[0]
print(f"\nquestion:  {p['question'][:120]}")
print(f"evidence:  {p['evidence'][:120]}")
print(f"sql:       {p['sql'][:120].replace(chr(10), ' ')}...")
print(f"expected_output: {str(p['expected_output'])[:80]}")

db-9 pairs: 30

question:  Can you show me a multi-carrier rate comparison that includes zone analysis and identifies cost optimization opportuniti
evidence:  The query constructs five CTEs: package_dimensions (billable weight, DIM divisor 166), zone_lookup (origin-destination z
sql:       WITH package_dimensions AS (     -- First CTE: Calculate package dimensions and dimensional weight     SELECT         p....
expected_output: Rate comparison results showing cheapest carrier, fastest carrier, cost savings 


## 3. Field coverage across the corpus

In [4]:
fields = ["question", "sql", "description", "evidence", "expected_output"]
cov = {f: 0 for f in fields}
total = 0
for n, m in mounted.items():
    for p in get_bird_pairs(m, n):
        total += 1
        for f in fields:
            if p.get(f):
                cov[f] += 1
print(f"{'field':<17}{'coverage':>10}")
for f in fields:
    print(f"{f:<17}{cov[f]:>6}/{total}")

field              coverage
question            480/480
sql                 480/480
description         480/480
evidence            480/480
expected_output     480/480


## 4. Next steps (live execution)

1. Start containers: `cd client && ./scripts/setup_docker.sh -a` (or `--schema-only` for the quick test).
2. Execute agent-generated SQL against `localhost:<port>` per the table above.
3. Reward: execution success + result match vs the gold `sql` output
   (`expected_output` is a prose description — materialize real result sets as the oracle;
   see `docs/TECHNICAL_EXECUTION.md` §2 and `results/gold_query_execution_20260805.json`).